# Aula 13 — Métricas e Indicadores: da fórmula à decisão

**Aprender a interpretar números antes de aprender a celebrar números**

Nesta aula, o foco não é apenas calcular métricas. É aprender a **ler o comportamento de um modelo a partir delas**.

Você vai trabalhar com cenários em que recebe valores de indicadores, matrizes de confusão, thresholds e custos de erro — e precisa decidir o que esses números realmente significam.


## 1. Objetivos de aprendizagem

Ao final desta aula, você deverá ser capaz de:

- interpretar `accuracy`, `precision`, `recall`, `F1-score`, `specificity` e `balanced accuracy`;
- reconhecer quando uma métrica alta esconde um problema grave;
- distinguir falso positivo de falso negativo em linguagem de negócio;
- interpretar `macro`, `weighted` e `micro` averages;
- compreender o efeito do threshold de decisão;
- relacionar métricas técnicas ao custo do erro;
- combinar métricas de qualidade com indicadores operacionais;
- justificar uma decisão de modelo usando evidência.


## 📘 Palavras-chave da aula

Use o Glossário Vivo como apoio ativo durante a aula:

**[Matriz de confusão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#matriz-de-confusão) · [Acurácia](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#acurácia) · [Precisão](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#precisão) · [Recall](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#recall) · [F1-score](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#f1-score) · [Falso positivo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-positivo) · [Falso negativo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-negativo) · [Especificidade](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#especificidade) · [Balanced Accuracy](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#balanced-accuracy) · [Support](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#support) · [Threshold](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#threshold-de-decisão) · [Abstenção](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#taxa-de-abstenção)**

> Consulte os termos quando surgirem nos exercícios. A intenção é transformar o glossário em instrumento de navegação conceitual, não apenas em referência final.


## Antes de começar — trabalhe na sua própria cópia

Crie uma cópia do notebook no Kaggle. Esta aula foi desenhada para você **responder, comparar sua interpretação e só depois revelar a solução**.


## 2. Métricas contam histórias sobre erros

Um número isolado raramente é suficiente.

Pense assim:

```text
métrica
→ resume um comportamento
→ destaca alguns erros
→ esconde outros
```

A pergunta madura não é:

> Qual métrica é maior?

Mas:

> **Qual erro essa métrica está tornando visível — e qual erro ela pode estar escondendo?**


## 3. A matriz de confusão como mapa dos erros

Antes das fórmulas, precisamos dominar quatro elementos:

| | Predito positivo | Predito negativo |
|---|---:|---:|
| **Real positivo** | TP | FN |
| **Real negativo** | FP | TN |

- **TP**: detectou corretamente o positivo;
- **FN**: deixou escapar um positivo real;
- **FP**: acusou positivo onde não havia;
- **TN**: rejeitou corretamente um negativo.

> 📘 Consulte: [Falso positivo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-positivo) e [Falso negativo](https://github.com/pedroregato/text-intelligence-lab/blob/main/docs/glossary/glossary.pt-BR.md#falso-negativo).


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Exemplo visual de matriz de confusão.
# O objetivo não é avaliar um modelo real, mas treinar a leitura das quatro regiões.
cm = np.array([[870, 30],
               [70, 30]])

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm)
ax.set_xticks([0, 1], labels=["Negativo", "Positivo"])
ax.set_yticks([0, 1], labels=["Negativo", "Positivo"])
ax.set_xlabel("Predito")
ax.set_ylabel("Real")
ax.set_title("Matriz de confusão — exemplo didático")

for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center")

plt.show()


## 4. Desafio 1 — A armadilha da acurácia

Considere um sistema de detecção de fraude com 1.000 transações:

- 950 legítimas;
- 50 fraudulentas.

O modelo prevê **todas** como legítimas.

### Perguntas
1. Qual é a acurácia?
2. Qual é o recall da classe fraude?
3. Você colocaria esse modelo em produção?


In [ ]:
# Responda antes de revelar a solução.
accuracy_guess = None
fraud_recall_guess = None
production_decision = ""


### Solução comentada

- acurácia = `950 / 1000 = 0.95`;
- recall da fraude = `0 / 50 = 0.00`;
- apesar de 95% de acurácia, o sistema **não detecta nenhuma fraude**.

> **Lição:** em classes raras, acurácia pode ser uma métrica confortável e enganosa.


## 5. Precision e Recall contam histórias diferentes

### Precision
Entre os casos que o modelo chamou de positivos, quantos realmente eram positivos?

### Recall
Entre todos os positivos reais, quantos o modelo conseguiu encontrar?

Uma forma útil de lembrar:

```text
precision → posso confiar nos alertas?
recall    → estou deixando casos importantes escapar?
```


## 6. Desafio 2 — Interprete apenas os valores

Um modelo apresenta:

```text
precision = 0.92
recall    = 0.48
```

Qual interpretação é mais adequada?

A. O modelo encontra quase todos os positivos.
B. Quando alerta, costuma acertar, mas deixa muitos positivos escaparem.
C. O modelo produz muitos falsos positivos.
D. Não é possível dizer nada.


In [ ]:
answer_challenge_2 = ""
print("Sua resposta:", answer_challenge_2)


### Interpretação

A resposta esperada é **B**.

Precision alta sugere poucos falsos positivos entre os alertas emitidos. Recall baixo indica muitos falsos negativos.

> A mesma combinação pode ser excelente ou péssima dependendo do custo de perder um positivo.


## 7. F1-score: equilíbrio, não mágica

O F1-score combina precision e recall por média harmônica.

Ele é útil quando queremos equilíbrio entre os dois, mas não conhece o custo real do negócio.

Exemplo:

| Modelo | Precision | Recall | F1 |
|---|---:|---:|---:|
| A | 0.91 | 0.52 | 0.66 |
| B | 0.72 | 0.83 | 0.77 |

O Modelo B tem F1 maior. Mas isso não encerra a decisão.


## 8. Desafio 3 — Qual modelo escolher?

### Cenário A — triagem de doença grave
Perder um caso positivo é muito caro.

### Cenário B — revisão manual cara
Cada falso alerta consome 40 minutos de um especialista.

Com os valores da tabela anterior, qual modelo tende a ser mais atraente em cada cenário? Explique usando **precision e recall**, não apenas F1.


In [ ]:
scenario_a_choice = ""
scenario_b_choice = ""
justification = ""


### Leitura esperada

- no cenário A, recall tende a ganhar importância;
- no cenário B, precision tende a ganhar importância;
- F1 ajuda a resumir, mas **não substitui a função de custo do problema**.


## 9. Macro, weighted e micro: quando a média muda a história

Considere três classes:

| Classe | Support | F1 |
|---|---:|---:|
| comum | 900 | 0.95 |
| importante | 80 | 0.60 |
| crítica | 20 | 0.20 |

A classe crítica é pequena e mal atendida.

- **macro average** dá o mesmo peso às classes;
- **weighted average** pondera pelo support;
- **micro average** agrega decisões individuais antes de calcular a métrica.


In [ ]:
supports = np.array([900, 80, 20])
f1_by_class = np.array([0.95, 0.60, 0.20])

macro_f1 = f1_by_class.mean()
weighted_f1 = np.average(f1_by_class, weights=supports)

print("Macro F1 aproximado   :", round(macro_f1, 3))
print("Weighted F1 aproximado:", round(weighted_f1, 3))


### O que observar

O weighted F1 pode parecer confortável porque a classe majoritária domina a média.

O macro F1 expõe com mais força a fragilidade da classe crítica.

> **Média alta não significa desempenho uniforme entre classes.**


## 10. Especificidade e Balanced Accuracy

Recall olha para os positivos. **Specificity** olha para os negativos corretamente rejeitados.

A **Balanced Accuracy** calcula, no caso binário, a média entre recall e specificity.

Ela é especialmente útil quando as classes são desbalanceadas.


## 11. Threshold: a decisão não termina no modelo

Muitos classificadores produzem um score ou probabilidade e depois aplicam um threshold.

```text
score >= threshold → positivo
score <  threshold → negativo
```

Alterar esse limite muda a relação entre precision e recall.


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd

y_true = np.array([1,1,1,1,1,0,0,0,0,0,0,0])
scores = np.array([.95,.82,.68,.54,.41,.78,.62,.49,.35,.28,.12,.05])

rows = []
for threshold in [0.30, 0.50, 0.70]:
    y_pred = (scores >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "positivos_preditos": int(y_pred.sum()),
    })

threshold_results = pd.DataFrame(rows)
threshold_results.round(3)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(threshold_results["threshold"], threshold_results["precision"], marker="o", label="precision")
ax.plot(threshold_results["threshold"], threshold_results["recall"], marker="o", label="recall")
ax.plot(threshold_results["threshold"], threshold_results["f1"], marker="o", label="f1")
ax.set_xlabel("Threshold")
ax.set_ylabel("Valor")
ax.set_ylim(0, 1.05)
ax.set_title("Como o threshold muda o comportamento")
ax.legend()
plt.show()


### Pergunta de interpretação

Se o custo de um falso negativo aumentar muito, você tenderia a **subir ou baixar** o threshold?

A resposta esperada, em geral, é **baixar** o threshold para aumentar recall — aceitando possivelmente mais falsos positivos.


## 12. Custos: quando o erro vira dinheiro, tempo ou risco

Até aqui tratamos erros como contagens e métricas. Em sistemas reais, cada erro pode ter um impacto diferente.

Por isso, uma avaliação madura precisa responder:

> **quanto custa cada tipo de decisão?**

A ideia central é transformar erros técnicos em consequências mensuráveis.


### Tipos de custo que devemos considerar

| Tipo de custo | Exemplo |
|---|---|
| **Falso positivo** | bloquear uma transação legítima; encaminhar um caso desnecessário para revisão |
| **Falso negativo** | deixar passar uma fraude; não detectar uma reclamação crítica |
| **Revisão humana** | tempo de analista por caso encaminhado |
| **Retrabalho** | corrigir classificações erradas depois da automação |
| **Latência** | perda de experiência do usuário ou SLA |
| **Computacional** | CPU/GPU, memória, storage, chamadas de API |
| **Infraestrutura** | servidores, filas, observabilidade, logging, escalabilidade |
| **Oportunidade** | receita perdida por uma decisão lenta ou errada |
| **Regulatório** | multas, sanções, auditorias, obrigações de correção |
| **Reputacional** | perda de confiança do cliente ou da organização |

> Nem todo custo é financeiro de forma imediata. Tempo, risco, experiência e reputação também podem ser convertidos em impacto econômico ou operacional.


### Custo esperado de uma decisão

Uma forma simples de começar é atribuir um custo médio a cada tipo de erro:

```text
custo_total =
    FP × custo_FP
  + FN × custo_FN
  + custo_operacional
```

Podemos expandir:

```text
custo_total =
    FP × custo_FP
  + FN × custo_FN
  + revisões_humanas × custo_revisão
  + custo_infra
  + custo_latência
```

O objetivo não é obter uma contabilidade perfeita, e sim criar uma aproximação suficiente para comparar alternativas.


In [ ]:
# Exemplo simples de custo esperado por 10.000 decisões

FP = 180
FN = 40

cost_fp = 8.0     # R$ por falso positivo
cost_fn = 120.0   # R$ por falso negativo

human_reviews = 250
cost_review = 6.0

infra_cost = 900.0

total_cost = (
    FP * cost_fp
    + FN * cost_fn
    + human_reviews * cost_review
    + infra_cost
)

print(f"Custo total estimado: R$ {total_cost:,.2f}")
print(f"Custo médio por decisão: R$ {total_cost / 10000:,.4f}")


### Por que custo muda a escolha do modelo

Considere dois modelos:

| Indicador | Modelo A | Modelo B |
|---|---:|---:|
| FP | 300 | 120 |
| FN | 20 | 50 |
| custo por execução | R$ 0,01 | R$ 0,08 |

Se um falso negativo custa muito mais do que um falso positivo, o Modelo A pode ser preferível.

Se falsos positivos geram revisões caras e o custo computacional importa muito, o Modelo B pode se tornar mais atraente.

> **Não existe “melhor modelo” sem uma função de custo ou um objetivo de negócio.**


In [ ]:
def expected_cost(fp, fn, cost_fp, cost_fn, inference_cost, n_cases):
    return (
        fp * cost_fp
        + fn * cost_fn
        + inference_cost * n_cases
    )

n_cases = 10000

cost_a = expected_cost(
    fp=300,
    fn=20,
    cost_fp=10,
    cost_fn=200,
    inference_cost=0.01,
    n_cases=n_cases,
)

cost_b = expected_cost(
    fp=120,
    fn=50,
    cost_fp=10,
    cost_fn=200,
    inference_cost=0.08,
    n_cases=n_cases,
)

print(f"Modelo A: R$ {cost_a:,.2f}")
print(f"Modelo B: R$ {cost_b:,.2f}")


### Custos que devem ser medidos ao longo do ciclo de vida

Não avalie custo apenas no momento do treinamento.

Pense no ciclo completo:

```text
desenvolvimento
→ treinamento
→ inferência
→ monitoramento
→ revisão humana
→ manutenção
→ retreinamento
```

Alguns custos são recorrentes e podem superar o custo inicial do modelo.

Exemplos:
- chamadas de API por milhão de requisições;
- GPU ligada continuamente;
- horas de revisão manual;
- storage de logs;
- reprocessamento após erro;
- retreinamentos periódicos;
- custo de incidentes e indisponibilidade.


### Exercício de interpretação de custo

Considere:

```text
Modelo X
F1 = 0.86
FN = 60
FP = 120
custo por inferência = R$ 0,02

Modelo Y
F1 = 0.84
FN = 30
FP = 220
custo por inferência = R$ 0,01
```

Agora suponha:

```text
custo_FN = R$ 150
custo_FP = R$ 8
10.000 decisões
```

Pergunta:

> Qual modelo gera menor custo esperado?

Observe que o modelo com maior F1 pode não ser o economicamente mais interessante.


## 13. Métrica técnica versus indicador operacional

Um modelo pode ter bom F1 e ainda ser inviável em produção.

Além das métricas preditivas, acompanhe indicadores como:

| Dimensão | Exemplos |
|---|---|
| Qualidade | precision, recall, F1, balanced accuracy |
| Operação | latência média, p95, throughput |
| Cobertura | taxa de abstenção, % de casos automatizados |
| Negócio | custo do erro, economia, conversão, retrabalho |
| Estabilidade | drift, distribuição das classes, degradação temporal |

> **Modelo bom no laboratório ≠ sistema bom em produção.**


## 14. Desafio 4 — Dois modelos, dois perfis

| Indicador | Modelo A | Modelo B |
|---|---:|---:|
| F1 macro | 0.84 | 0.82 |
| Recall classe crítica | 0.61 | 0.79 |
| Latência p95 | 45 ms | 210 ms |
| Taxa de abstenção | 2% | 8% |
| Custo estimado por 10 mil casos | R$ 420 | R$ 690 |

Pergunta: **qual modelo é melhor?**

Resposta madura: ainda não sabemos. Precisamos do contexto operacional e do custo da classe crítica.


## 15. Laboratório de interpretação — sem código

Leia cada caso e escreva a conclusão em uma frase.

### Caso 1
`accuracy = 0.97`, `recall_classe_rara = 0.18`

### Caso 2
`precision = 0.41`, `recall = 0.93`

### Caso 3
`macro_f1 = 0.58`, `weighted_f1 = 0.89`

### Caso 4
`F1 = 0.82`, mas `latência p95 = 2.8 s`

### Caso 5
Modelo novo: `F1 = 0.851`; baseline: `F1 = 0.846`; novo modelo custa 12× mais.


In [ ]:
interpretations = {
    "caso_1": "",
    "caso_2": "",
    "caso_3": "",
    "caso_4": "",
    "caso_5": "",
}

interpretations


### Leituras esperadas

1. Acurácia alta pode estar mascarando falha severa na classe rara.
2. O modelo encontra quase todos os positivos, mas gera muitos falsos alertas.
3. Classes majoritárias provavelmente dominam o weighted F1; verifique desempenho por classe.
4. Qualidade preditiva boa pode ser incompatível com requisito operacional de latência.
5. O ganho de 0.005 em F1 pode não justificar 12× de custo sem evidência de valor adicional.


## 16. Exercício guiado — encontre a métrica que falta

Considere:

```text
TP = 72
FP = 18
FN = 28
TN = 882
```

Calcule:
1. accuracy;
2. precision;
3. recall;
4. specificity;
5. F1-score.

Depois escreva uma frase interpretando o comportamento do modelo.


In [ ]:
# Escreva sua solução aqui.
TP, FP, FN, TN = 72, 18, 28, 882

accuracy = None
precision = None
recall = None
specificity = None
f1 = None


### Dica e solução

Execute a célula abaixo uma vez. Depois use `q13.hint()` ou `q13.solution()` somente se precisar.


In [ ]:
from IPython.display import Markdown, display

class TILExercise:
    def __init__(self, hint_text, solution_text):
        self._hint_text = hint_text
        self._solution_text = solution_text

    def hint(self):
        display(Markdown(f"### Dica\n\n{self._hint_text}"))

    def solution(self):
        display(Markdown(f"### Solução\n\n{self._solution_text}"))

q13 = TILExercise(
    hint_text=(
        "Use: accuracy=(TP+TN)/(TP+TN+FP+FN), precision=TP/(TP+FP), "
        "recall=TP/(TP+FN), specificity=TN/(TN+FP) e "
        "F1=2*precision*recall/(precision+recall)."
    ),
    solution_text=(
        "```python\n"
        "accuracy = (TP + TN) / (TP + TN + FP + FN)\n"
        "precision = TP / (TP + FP)\n"
        "recall = TP / (TP + FN)\n"
        "specificity = TN / (TN + FP)\n"
        "f1 = 2 * precision * recall / (precision + recall)\n\n"
        "print(round(accuracy, 3), round(precision, 3), round(recall, 3), "
        "round(specificity, 3), round(f1, 3))\n"
        "```"
    ),
)

print("Exercício preparado.")


In [ ]:
# q13.hint()


In [ ]:
# q13.solution()


## 17. Checklist de interpretação

Antes de concluir que um modelo é bom, pergunte:

- qual classe é mais importante?
- qual erro custa mais: FP ou FN?
- há desbalanceamento?
- a média está escondendo uma classe ruim?
- o threshold está adequado ao objetivo?
- a métrica é estável entre folds/períodos?
- a latência é aceitável?
- existe abstention ou revisão humana?
- o ganho frente ao baseline justifica o custo?

Esse checklist é mais importante do que decorar qualquer fórmula isoladamente.


## 18. Reprodutibilidade

- Python;
- bibliotecas: `numpy`, `pandas`, `matplotlib`, `scikit-learn`;
- dataset externo: nenhum;
- internet: desabilitada;
- exemplos numéricos fixos;
- sem dependência de modelo externo;
- foco: interpretação de métricas e indicadores.


## 19. Resumo

Nesta aula, você aprendeu que:

- métricas são lentes sobre tipos diferentes de erro;
- acurácia pode enganar em classes raras;
- precision e recall respondem perguntas diferentes;
- F1 não conhece o custo do negócio;
- macro e weighted podem contar histórias muito diferentes;
- threshold altera o comportamento do sistema;
- qualidade preditiva deve ser combinada com indicadores operacionais;
- escolher um modelo é uma decisão técnica e econômica.

### Ideia principal

> **Interpretar métricas é compreender o comportamento do sistema antes de decidir se ele merece confiança.**

**Fim da Aula 13.**


## Próximos laboratórios da Aula 13

A interpretação das métricas continua em dois laboratórios complementares:

- **Aula 13B — Metric Scenario Lab:** thresholds, cenários e custos de erro;
- **Aula 13C — Model Routing, Orchestration e Utility:** quality gates, custo, latência e sistemas compostos de IA.

```text
métrica → custo do erro → threshold → cenário → routing → orchestration
```

A 13C também introduz a distinção entre **DEMO** e **EVIDENCE**: proxies didáticos são úteis para aprender, mas decisões arquiteturais reais exigem medições versionadas e comparáveis.
